# Updated AD Classifier (Methodology-Preserving)

This notebook keeps the **same core architecture** (dual 3D encoders + asymmetric cross-attention + multi-task heads),
and applies the requested upgrades in training/data handling/regularization:

- Transfer learning initialization for 3D backbones (MedicalNet/Med3D checkpoints)
- Clinical feature embedding (age, sex, education, APOE if present)
- Strong 3D medical augmentation (TorchIO)
- Label smoothing
- Dynamic multi-task loss weighting
- 5-fold cross-validation
- Longer training + early stopping
- Confidence penalty for calibration
- PET-type balancing in training batches

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, gc, math, time, copy, random, json, warnings, sys, subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, mean_absolute_error, mean_squared_error
)
from scipy.ndimage import zoom as scipy_zoom
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

# TorchIO for 3D medical augmentation (auto-install if needed)
try:
    import torchio as tio
except ImportError:
    print('Installing TorchIO...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchio'])
    import torchio as tio

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB')

In [ ]:
# ============================================================
# CONFIG
# ============================================================
class Config:
    # Paths
    DATA_ROOT = '/kaggle/input/nacc-preprocessed'
    CSV_PATH = '/kaggle/input/nacc-preprocessed/500_patients.csv'
    SAVE_DIR = '/kaggle/working/updated_ad_classifier'

    # PET files
    SYNTHETIC_PET_ROOT = ''
    SYNTHETIC_PET_FILE = 'pet_synthetic.npy'
    REAL_PET_FILE = 'pet_processed.npy'
    REAL_PET_CONFIDENCE = 1.0
    SYNTH_PET_CONFIDENCE = 0.3

    # Shapes
    CROP_SHAPE = (96, 112, 96)

    # Labels
    NUM_CLASSES = 3
    LABEL_MAP = {1: 0, 3: 1, 4: 2}
    CLASS_NAMES = ['CN', 'MCI', 'AD']
    MMSE_MISSING = [88, -4, 95, 96, 97, 98]

    # Clinical features (used as auxiliary tokens/embedding)
    CLINICAL_CANDIDATES = ['AGE', 'SEX', 'EDUC', 'APOE4']
    CLINICAL_EMBED_DIM = 64

    # Model
    FEATURE_DIM = 512
    FUSION_HEADS = 8
    FUSION_LAYERS = 2
    FUSION_DROPOUT = 0.1
    DROPOUT = 0.3

    # Training
    BATCH_SIZE = 2
    EPOCHS = 160
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    GRAD_CLIP = 1.0
    WARMUP_EPOCHS = 8
    EARLY_STOPPING_PATIENCE = 20

    # Transfer learning
    USE_PRETRAIN = True
    MED3D_MRI_CKPT = ''
    MED3D_PET_CKPT = ''
    FREEZE_BACKBONE_EPOCHS = 5

    # Losses
    LABEL_SMOOTHING = 0.1
    CLS_WEIGHT = 1.0
    REG_WEIGHT_START = 0.2
    REG_WEIGHT_END = 0.5
    CONFIDENCE_PENALTY = 0.01

    # CV
    N_FOLDS = 5

    # MC Dropout + Calibration
    MC_SAMPLES = 30
    TEMP_LR = 0.01
    TEMP_ITERS = 50

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
print('Save dir:', cfg.SAVE_DIR)

In [ ]:
# ============================================================
# DATA PREP + CLINICAL FEATURES
# ============================================================
def center_crop_3d(volume, target_shape):
    d, h, w = volume.shape
    td, th, tw = target_shape
    sd = max(0, (d - td) // 2)
    sh = max(0, (h - th) // 2)
    sw = max(0, (w - tw) // 2)
    cropped = volume[sd:sd+td, sh:sh+th, sw:sw+tw]
    if cropped.shape != target_shape:
        padded = np.full(target_shape, -1.0, dtype=np.float32)
        pd_ = (td - cropped.shape[0]) // 2
        ph_ = (th - cropped.shape[1]) // 2
        pw_ = (tw - cropped.shape[2]) // 2
        padded[pd_:pd_+cropped.shape[0], ph_:ph_+cropped.shape[1], pw_:pw_+cropped.shape[2]] = cropped
        return padded
    return cropped

def map_sex_value(v):
    if pd.isna(v):
        return 0.0
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {'m', 'male', '1'}:
            return 1.0
        if s in {'f', 'female', '0', '2'}:
            return 0.0
    try:
        iv = float(v)
        if iv in [1.0]:
            return 1.0
        if iv in [0.0, 2.0]:
            return 0.0
    except Exception:
        pass
    return 0.0

def _find_pet_volume(naccid, data_root, cfg):
    real_path = os.path.join(data_root, naccid, cfg.REAL_PET_FILE)
    if os.path.exists(real_path):
        return real_path, False

    synth_main = os.path.join(data_root, naccid, cfg.SYNTHETIC_PET_FILE)
    if os.path.exists(synth_main):
        return synth_main, True

    if cfg.SYNTHETIC_PET_ROOT:
        p1 = os.path.join(cfg.SYNTHETIC_PET_ROOT, naccid, cfg.SYNTHETIC_PET_FILE)
        p2 = os.path.join(cfg.SYNTHETIC_PET_ROOT, naccid, cfg.REAL_PET_FILE)
        if os.path.exists(p1):
            return p1, True
        if os.path.exists(p2):
            return p2, True

    return None, None

def load_patient_labels_with_clinical(cfg):
    df = pd.read_csv(cfg.CSV_PATH)
    df = df[df['NACCUDSD'].isin(cfg.LABEL_MAP.keys())].copy()
    df['label'] = df['NACCUDSD'].map(cfg.LABEL_MAP).astype(int)

    df['mmse_valid'] = ~df['NACCMMSE'].isin(cfg.MMSE_MISSING)
    df['mmse_norm'] = df['NACCMMSE'].clip(0, 30) / 30.0
    df.loc[~df['mmse_valid'], 'mmse_norm'] = -1.0

    # Build clinical columns robustly from available schema
    candidates = {c.upper(): c for c in df.columns}
    age_col = candidates.get('AGE') or candidates.get('NACCAGE')
    sex_col = candidates.get('SEX') or candidates.get('NACCSEX')
    educ_col = candidates.get('EDUC') or candidates.get('NACCEDUC')
    apoe_col = candidates.get('APOE4') or candidates.get('NACCAPOE')

    df['clin_age'] = pd.to_numeric(df[age_col], errors='coerce') if age_col else np.nan
    df['clin_sex'] = df[sex_col].apply(map_sex_value) if sex_col else 0.0
    df['clin_educ'] = pd.to_numeric(df[educ_col], errors='coerce') if educ_col else np.nan
    if apoe_col:
        df['clin_apoe4'] = pd.to_numeric(df[apoe_col], errors='coerce').fillna(0.0)
    else:
        df['clin_apoe4'] = 0.0

    # Median fill + z-score standardization for continuous vars
    for c in ['clin_age', 'clin_educ']:
        med = df[c].median() if df[c].notna().any() else 0.0
        df[c] = df[c].fillna(med)
        std = df[c].std() if df[c].std() > 1e-8 else 1.0
        df[c] = (df[c] - df[c].mean()) / std

    # APOE4 as clipped count then standardize lightly
    df['clin_apoe4'] = df['clin_apoe4'].clip(0, 2)
    df['clin_apoe4'] = (df['clin_apoe4'] - df['clin_apoe4'].mean()) / (df['clin_apoe4'].std() + 1e-8)

    clinical_cols = ['clin_age', 'clin_sex', 'clin_educ', 'clin_apoe4']
    print(f'Loaded {len(df)} labeled patients')
    print('Class counts:', dict(df['label'].value_counts().sort_index()))
    print('Clinical columns used:', clinical_cols)
    return df, clinical_cols

patient_df, clinical_cols = load_patient_labels_with_clinical(cfg)

In [ ]:
# ============================================================
# DATASET + TORCHIO AUGMENTATION + BALANCED SAMPLER
# ============================================================
def build_torchio_augmenter():
    return tio.Compose([
        tio.RandomAffine(scales=(0.95, 1.05), degrees=8, translation=4, p=0.7),
        tio.RandomElasticDeformation(num_control_points=5, max_displacement=4.5, p=0.25),
        tio.RandomGamma(log_gamma=(-0.2, 0.2), p=0.5),
        tio.RandomBiasField(coefficients=0.4, p=0.35),
        tio.RandomNoise(mean=0.0, std=(0, 0.03), p=0.4),
        tio.RandomMotion(degrees=5, translation=3, num_transforms=2, image_interpolation='linear', p=0.2),
    ])

class ADClassificationDataset(Dataset):
    def __init__(self, patient_df, data_root, cfg, clinical_cols, augment=False):
        self.cfg = cfg
        self.augment = augment
        self.clinical_cols = clinical_cols
        self.samples = []
        self.aug = build_torchio_augmenter() if augment else None

        found_real, found_synth, missing = 0, 0, 0
        for _, row in patient_df.iterrows():
            naccid = str(row['NACCID'])
            mri_p = os.path.join(data_root, naccid, 'mri_processed.npy')
            if not os.path.exists(mri_p):
                missing += 1
                continue

            pet_p, is_synthetic = _find_pet_volume(naccid, data_root, cfg)
            if pet_p is None:
                missing += 1
                continue

            pet_conf = cfg.SYNTH_PET_CONFIDENCE if is_synthetic else cfg.REAL_PET_CONFIDENCE
            clin = row[clinical_cols].values.astype(np.float32)

            self.samples.append({
                'mri_path': mri_p,
                'pet_path': pet_p,
                'label': int(row['label']),
                'mmse': float(row['mmse_norm']),
                'mmse_valid': bool(row['mmse_valid']),
                'pet_conf': float(pet_conf),
                'pet_type': int(is_synthetic),  # 0=real, 1=synthetic
                'clinical': clin
            })

            if is_synthetic:
                found_synth += 1
            else:
                found_real += 1

        print(f'Dataset built: real={found_real}, synth={found_synth}, missing={missing}')

    def __len__(self):
        return len(self.samples)

    def get_labels(self):
        return [s['label'] for s in self.samples]

    def get_pet_types(self):
        return [s['pet_type'] for s in self.samples]

    def __getitem__(self, idx):
        s = self.samples[idx]
        mri = np.load(s['mri_path'], mmap_mode='r').astype(np.float32)
        pet = np.load(s['pet_path'], mmap_mode='r').astype(np.float32)

        mri = center_crop_3d(np.array(mri), self.cfg.CROP_SHAPE)
        pet = center_crop_3d(np.array(pet), self.cfg.CROP_SHAPE)

        if self.augment:
            subject = tio.Subject(
                mri=tio.ScalarImage(tensor=torch.from_numpy(mri).unsqueeze(0)),
                pet=tio.ScalarImage(tensor=torch.from_numpy(pet).unsqueeze(0)),
            )
            aug_subject = self.aug(subject)
            mri = aug_subject.mri.tensor.squeeze(0).numpy().astype(np.float32)
            pet = aug_subject.pet.tensor.squeeze(0).numpy().astype(np.float32)

        return {
            'mri': torch.from_numpy(mri).unsqueeze(0),
            'pet': torch.from_numpy(pet).unsqueeze(0),
            'label': torch.tensor(s['label'], dtype=torch.long),
            'mmse': torch.tensor(s['mmse'], dtype=torch.float32),
            'mmse_valid': torch.tensor(s['mmse_valid'], dtype=torch.bool),
            'pet_confidence': torch.tensor(s['pet_conf'], dtype=torch.float32),
            'pet_type': torch.tensor(s['pet_type'], dtype=torch.long),
            'clinical': torch.tensor(s['clinical'], dtype=torch.float32),
        }

def build_balanced_sampler(dataset):
    labels = dataset.get_labels()
    pet_types = dataset.get_pet_types()

    class_counts = Counter(labels)
    pet_counts = Counter(pet_types)

    weights = []
    for y, p in zip(labels, pet_types):
        w_class = 1.0 / class_counts[y]
        w_pet = 1.0 / pet_counts[p]
        weights.append(w_class * w_pet)

    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

In [ ]:
# ============================================================
# MODEL (UNCHANGED CORE ARCHITECTURE) + CLINICAL EMBEDDING
# ============================================================
class BasicBlock3D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.conv2 = nn.Conv3d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm3d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNet3D(nn.Module):
    def __init__(self, in_channels=1, num_blocks=(2, 2, 2, 2)):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv3d(in_channels, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm3d(64)
        self.maxpool = nn.MaxPool3d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64,  num_blocks[0], stride=1)
        self.layer2 = self._make_layer(128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool3d(1)

    def _make_layer(self, planes, num_blocks, stride):
        layers = [BasicBlock3D(self.in_planes, planes, stride)]
        self.in_planes = planes
        for _ in range(1, num_blocks):
            layers.append(BasicBlock3D(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        spatial = self.layer4(x)
        pooled = self.avgpool(spatial).flatten(1)
        return pooled, spatial

class AsymmetricCrossAttentionLayer(nn.Module):
    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.mri_to_pet = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.pet_to_mri = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_mri = nn.LayerNorm(dim)
        self.norm_pet = nn.LayerNorm(dim)
        self.gate = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid())
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout),
        )
        self.norm_out = nn.LayerNorm(dim)

    def forward(self, mri_tokens, pet_tokens, pet_confidence=None):
        if pet_confidence is not None:
            pc = pet_confidence.view(-1, 1, 1)
            scaled_pet = pet_tokens * pc
        else:
            scaled_pet = pet_tokens

        mri_attended, _ = self.mri_to_pet(query=mri_tokens, key=scaled_pet, value=scaled_pet)
        mri_attended = self.norm_mri(mri_tokens + mri_attended)

        pet_attended, attn_w = self.pet_to_mri(query=scaled_pet, key=mri_tokens, value=mri_tokens)
        pet_attended = self.norm_pet(scaled_pet + pet_attended)

        raw_gate = self.gate(torch.cat([mri_attended.mean(1), pet_attended.mean(1)], dim=-1))
        if pet_confidence is not None:
            pc = pet_confidence.view(-1, 1)
            gate = raw_gate * pc + (1.0 - pc)
        else:
            gate = raw_gate

        fused = gate.unsqueeze(1) * mri_attended + (1 - gate.unsqueeze(1)) * pet_attended
        fused = self.norm_out(fused + self.ffn(fused))
        return fused, attn_w

class AsymmetricCrossAttentionFusion(nn.Module):
    def __init__(self, feature_dim=512, num_heads=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            AsymmetricCrossAttentionLayer(feature_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, mri_spatial, pet_spatial, pet_confidence=None):
        mri_tokens = mri_spatial.flatten(2).permute(0, 2, 1)
        pet_tokens = pet_spatial.flatten(2).permute(0, 2, 1)
        attn_weights = None
        for layer in self.layers:
            mri_tokens, attn_weights = layer(mri_tokens, pet_tokens, pet_confidence)
            pet_tokens = mri_tokens
        fused = self.pool(mri_tokens.permute(0, 2, 1)).squeeze(-1)
        return fused, attn_weights

class ADClassificationModel(nn.Module):
    def __init__(self, cfg, clinical_dim):
        super().__init__()
        self.mri_backbone = ResNet3D(in_channels=1)
        self.pet_backbone = ResNet3D(in_channels=1)
        self.fusion = AsymmetricCrossAttentionFusion(
            feature_dim=cfg.FEATURE_DIM,
            num_heads=cfg.FUSION_HEADS,
            num_layers=cfg.FUSION_LAYERS,
            dropout=cfg.FUSION_DROPOUT
        )

        # Clinical token/embedding branch
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(64, cfg.CLINICAL_EMBED_DIM),
            nn.ReLU(inplace=True),
        )

        combined_dim = cfg.FEATURE_DIM + cfg.CLINICAL_EMBED_DIM
        self.cls_head = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(256, cfg.NUM_CLASSES),
        )
        self.reg_head = nn.Sequential(
            nn.Linear(combined_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, mri, pet, clinical, pet_confidence=None):
        _, mri_spatial = self.mri_backbone(mri)
        _, pet_spatial = self.pet_backbone(pet)
        fused_img, attn_w = self.fusion(mri_spatial, pet_spatial, pet_confidence)
        clin = self.clinical_encoder(clinical)
        fused = torch.cat([fused_img, clin], dim=1)
        cls_logits = self.cls_head(fused)
        mmse_pred = self.reg_head(fused)
        return cls_logits, mmse_pred, attn_w

    def enable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

In [ ]:
# ============================================================
# TRANSFER LEARNING UTILITIES
# ============================================================
def _clean_state_dict_keys(sd):
    cleaned = {}
    for k, v in sd.items():
        nk = k
        if nk.startswith('module.'):
            nk = nk[len('module.'):]
        cleaned[nk] = v
    return cleaned

def load_pretrained_backbone(backbone, ckpt_path):
    if not ckpt_path or not os.path.exists(ckpt_path):
        print(f'[INFO] Pretrain checkpoint not found: {ckpt_path}')
        return

    ckpt = torch.load(ckpt_path, map_location='cpu')
    state = ckpt.get('state_dict', ckpt)
    state = _clean_state_dict_keys(state)

    # Keep only matching backbone keys
    own = backbone.state_dict()
    loaded = {}
    for k, v in state.items():
        if k in own and own[k].shape == v.shape:
            loaded[k] = v

    missing, unexpected = backbone.load_state_dict(loaded, strict=False)
    print(f'[INFO] Loaded pretrained keys: {len(loaded)} | missing={len(missing)} | unexpected={len(unexpected)}')

def set_backbone_trainable(model, trainable):
    for p in model.mri_backbone.parameters():
        p.requires_grad = trainable
    for p in model.pet_backbone.parameters():
        p.requires_grad = trainable

In [ ]:
# ============================================================
# TRAIN / EVAL UTILS
# ============================================================
def get_lr_lambda(epoch, warmup_epochs, total_epochs):
    if epoch < warmup_epochs:
        return (epoch + 1) / max(1, warmup_epochs)
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

def reg_weight_schedule(epoch, total_epochs, start_w, end_w):
    alpha = min(1.0, max(0.0, epoch / max(1, total_epochs - 1)))
    return start_w + alpha * (end_w - start_w)

def confidence_penalty_loss(logits):
    probs = F.softmax(logits, dim=1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=1)
    return -entropy.mean()

def train_one_epoch(model, loader, optimizer, cls_criterion, cfg, epoch, scaler=None):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    reg_w = reg_weight_schedule(epoch - 1, cfg.EPOCHS, cfg.REG_WEIGHT_START, cfg.REG_WEIGHT_END)

    for batch in tqdm(loader, desc='Train', leave=False):
        mri = batch['mri'].to(device, non_blocking=True)
        pet = batch['pet'].to(device, non_blocking=True)
        clin = batch['clinical'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        mmse = batch['mmse'].to(device, non_blocking=True)
        mmse_valid = batch['mmse_valid'].to(device, non_blocking=True)
        pet_conf = batch['pet_confidence'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda', enabled=(device.type == 'cuda')):
            cls_logits, mmse_pred, _ = model(mri, pet, clin, pet_conf)
            l_cls = cls_criterion(cls_logits, labels)
            if mmse_valid.any():
                l_reg = F.smooth_l1_loss(mmse_pred[mmse_valid].squeeze(-1), mmse[mmse_valid])
            else:
                l_reg = torch.tensor(0.0, device=device)

            conf_pen = confidence_penalty_loss(cls_logits)
            loss = cfg.CLS_WEIGHT * l_cls + reg_w * l_reg + cfg.CONFIDENCE_PENALTY * conf_pen

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

        total_loss += loss.item()
        preds = cls_logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_n += labels.size(0)

    return {
        'loss': total_loss / max(1, len(loader)),
        'acc': total_correct / max(1, total_n),
        'reg_w': reg_w,
    }

@torch.no_grad()
def evaluate(model, loader, cls_criterion):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    all_mmse_pred, all_mmse_true, all_mmse_valid = [], [], []
    total_loss = 0.0

    for batch in tqdm(loader, desc='Eval', leave=False):
        mri = batch['mri'].to(device, non_blocking=True)
        pet = batch['pet'].to(device, non_blocking=True)
        clin = batch['clinical'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        mmse = batch['mmse'].to(device, non_blocking=True)
        mmse_valid = batch['mmse_valid'].to(device, non_blocking=True)
        pet_conf = batch['pet_confidence'].to(device, non_blocking=True)

        with autocast('cuda', enabled=(device.type == 'cuda')):
            cls_logits, mmse_pred, _ = model(mri, pet, clin, pet_conf)
            total_loss += cls_criterion(cls_logits, labels).item()

        probs = F.softmax(cls_logits, dim=1)
        preds = cls_logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        all_mmse_pred.extend(mmse_pred.squeeze(-1).cpu().numpy())
        all_mmse_true.extend(mmse.cpu().numpy())
        all_mmse_valid.extend(mmse_valid.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_mmse_pred = np.array(all_mmse_pred)
    all_mmse_true = np.array(all_mmse_true)
    all_mmse_valid = np.array(all_mmse_valid, dtype=bool)

    mmse_mae = -1.0
    if all_mmse_valid.any():
        mmse_mae = mean_absolute_error(all_mmse_true[all_mmse_valid] * 30.0, all_mmse_pred[all_mmse_valid] * 30.0)

    return {
        'loss': total_loss / max(1, len(loader)),
        'acc': accuracy_score(all_labels, all_preds),
        'bal_acc': balanced_accuracy_score(all_labels, all_preds),
        'mmse_mae': mmse_mae,
        'preds': all_preds,
        'labels': all_labels,
        'probs': all_probs,
    }

In [ ]:
# ============================================================
# K-FOLD TRAINING (5-FOLD) + EARLY STOPPING
# ============================================================
def run_fold_training(fold_id, train_df, val_df, cfg, clinical_cols):
    train_dataset = ADClassificationDataset(train_df, cfg.DATA_ROOT, cfg, clinical_cols, augment=True)
    val_dataset = ADClassificationDataset(val_df, cfg.DATA_ROOT, cfg, clinical_cols, augment=False)

    train_sampler = build_balanced_sampler(train_dataset)
    train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, sampler=train_sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = ADClassificationModel(cfg, clinical_dim=len(clinical_cols)).to(device)

    # Optional pretraining init
    if cfg.USE_PRETRAIN:
        load_pretrained_backbone(model.mri_backbone, cfg.MED3D_MRI_CKPT)
        load_pretrained_backbone(model.pet_backbone, cfg.MED3D_PET_CKPT)

    # Freeze backbones for warm start
    if cfg.FREEZE_BACKBONE_EPOCHS > 0:
        set_backbone_trainable(model, False)

    # Label smoothing
    cls_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda ep: get_lr_lambda(ep, cfg.WARMUP_EPOCHS, cfg.EPOCHS))
    scaler = GradScaler('cuda') if device.type == 'cuda' else None

    best_metric = -1.0
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(1, cfg.EPOCHS + 1):
        if epoch == cfg.FREEZE_BACKBONE_EPOCHS + 1 and cfg.FREEZE_BACKBONE_EPOCHS > 0:
            set_backbone_trainable(model, True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda ep: get_lr_lambda(ep, cfg.WARMUP_EPOCHS, cfg.EPOCHS))

        train_stats = train_one_epoch(model, train_loader, optimizer, cls_criterion, cfg, epoch=epoch, scaler=scaler)
        scheduler.step()
        val_stats = evaluate(model, val_loader, cls_criterion)

        score = val_stats['bal_acc']
        history.append({
            'epoch': epoch,
            'train_loss': train_stats['loss'],
            'train_acc': train_stats['acc'],
            'val_loss': val_stats['loss'],
            'val_acc': val_stats['acc'],
            'val_bal_acc': val_stats['bal_acc'],
            'val_mmse_mae': val_stats['mmse_mae'],
            'reg_w': train_stats['reg_w'],
        })

        print(
            f"[Fold {fold_id}] Epoch {epoch:03d} | "
            f"tr_loss={train_stats['loss']:.4f} | "
            f"val_bal_acc={val_stats['bal_acc']:.4f} | "
            f"mmse_mae={val_stats['mmse_mae']:.2f} | "
            f"reg_w={train_stats['reg_w']:.3f}"
        )

        if score > best_metric:
            best_metric = score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= cfg.EARLY_STOPPING_PATIENCE:
            print(f'[Fold {fold_id}] Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    final_val = evaluate(model, val_loader, cls_criterion)

    fold_ckpt = os.path.join(cfg.SAVE_DIR, f'best_fold_{fold_id}.pt')
    torch.save({
        'fold': fold_id,
        'model_state_dict': model.state_dict(),
        'val_metrics': final_val,
        'history': history
    }, fold_ckpt)

    return model, history, final_val

skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=SEED)
X = np.arange(len(patient_df))
y = patient_df['label'].values

fold_summaries = []
best_fold_id = None
best_fold_bal_acc = -1
best_fold_model = None
best_fold_val_df = None

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    print('\n' + '=' * 80)
    print(f'FOLD {fold}/{cfg.N_FOLDS}')
    print('=' * 80)

    tr_df = patient_df.iloc[tr_idx].reset_index(drop=True)
    va_df = patient_df.iloc[va_idx].reset_index(drop=True)

    model_fold, history_fold, val_metrics = run_fold_training(fold, tr_df, va_df, cfg, clinical_cols)

    fold_summaries.append({
        'fold': fold,
        'acc': float(val_metrics['acc']),
        'bal_acc': float(val_metrics['bal_acc']),
        'mmse_mae': float(val_metrics['mmse_mae']),
    })

    if val_metrics['bal_acc'] > best_fold_bal_acc:
        best_fold_bal_acc = val_metrics['bal_acc']
        best_fold_id = fold
        best_fold_model = model_fold
        best_fold_val_df = va_df.copy()

accs = [f['acc'] for f in fold_summaries]
baccs = [f['bal_acc'] for f in fold_summaries]
mmse_maes = [f['mmse_mae'] for f in fold_summaries]

print('\n' + '=' * 80)
print('K-FOLD SUMMARY')
print('=' * 80)
for f in fold_summaries:
    print(f"Fold {f['fold']}: acc={f['acc']:.4f}, bal_acc={f['bal_acc']:.4f}, mmse_mae={f['mmse_mae']:.2f}")
print(f"Mean Accuracy:         {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
print(f"Mean Balanced Accuracy:{np.mean(baccs):.4f} +/- {np.std(baccs):.4f}")
print(f"Mean MMSE MAE:         {np.mean(mmse_maes):.2f} +/- {np.std(mmse_maes):.2f}")
print(f'Best fold: {best_fold_id} (bal_acc={best_fold_bal_acc:.4f})')

with open(os.path.join(cfg.SAVE_DIR, 'kfold_summary.json'), 'w') as f:
    json.dump({'folds': fold_summaries}, f, indent=2)

In [ ]:
# ============================================================
# MC DROPOUT + CALIBRATION ON BEST FOLD VALIDATION SPLIT
# ============================================================
@torch.no_grad()
def mc_dropout_inference(model, loader, n_samples=30):
    model.eval()
    model.enable_mc_dropout()

    all_mc_probs, all_labels = [], []

    for batch in tqdm(loader, desc='MC Dropout', leave=False):
        mri = batch['mri'].to(device)
        pet = batch['pet'].to(device)
        clin = batch['clinical'].to(device)
        pet_conf = batch['pet_confidence'].to(device)

        probs_s = []
        for _ in range(n_samples):
            with autocast('cuda', enabled=(device.type == 'cuda')):
                logits, _, _ = model(mri, pet, clin, pet_conf)
            probs_s.append(F.softmax(logits, dim=1).cpu().numpy())

        batch_mc = np.stack(probs_s, axis=1)
        all_mc_probs.append(batch_mc)
        all_labels.extend(batch['label'].numpy())

    all_mc_probs = np.concatenate(all_mc_probs, axis=0)
    all_labels = np.array(all_labels)
    mean_probs = all_mc_probs.mean(axis=1)
    preds = mean_probs.argmax(axis=1)

    entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-10), axis=1)
    expected_entropy = -np.mean(np.sum(all_mc_probs * np.log(all_mc_probs + 1e-10), axis=2), axis=1)
    mutual_info = entropy - expected_entropy

    return {
        'mc_probs': all_mc_probs,
        'mean_probs': mean_probs,
        'preds': preds,
        'labels': all_labels,
        'entropy': entropy,
        'mutual_info': mutual_info,
    }

class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

def compute_ece(probs, labels, n_bins=15):
    conf = np.max(probs, axis=1)
    pred = np.argmax(probs, axis=1)
    acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        if m.sum() > 0:
            ece += (m.sum() / len(labels)) * abs(acc[m].mean() - conf[m].mean())
    return ece

# Build validation loader from best fold for uncertainty/calibration analysis
best_val_dataset = ADClassificationDataset(best_fold_val_df, cfg.DATA_ROOT, cfg, clinical_cols, augment=False)
best_val_loader = DataLoader(best_val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

mc_results = mc_dropout_inference(best_fold_model, best_val_loader, n_samples=cfg.MC_SAMPLES)
mc_acc = accuracy_score(mc_results['labels'], mc_results['preds'])
mc_bal = balanced_accuracy_score(mc_results['labels'], mc_results['preds'])
ece_before = compute_ece(mc_results['mean_probs'], mc_results['labels'])

print(f'MC Accuracy: {mc_acc:.4f}')
print(f'MC Balanced Accuracy: {mc_bal:.4f}')
print(f"Mean Entropy: {mc_results['entropy'].mean():.4f}")
print(f"Mean Mutual Information: {mc_results['mutual_info'].mean():.4f}")
print(f'ECE before temperature scaling: {ece_before:.4f}')

In [ ]:
# ============================================================
# QUICK XAI: GRAD-CAM + ATTENTION MAPS (BEST FOLD MODEL)
# ============================================================
class GradCAM3D:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, mri, pet, clinical, pet_conf):
        self.model.eval()
        logits, _, _ = self.model(mri, pet, clinical, pet_conf)
        pred = logits.argmax(dim=1).item()
        self.model.zero_grad()
        logits[0, pred].backward(retain_graph=True)

        grads = self.gradients[0]
        acts = self.activations[0]
        weights = grads.mean(dim=(1, 2, 3))
        heatmap = torch.zeros_like(acts[0])
        for i, w in enumerate(weights):
            heatmap += w * acts[i]
        heatmap = F.relu(heatmap)
        heatmap = F.interpolate(heatmap[None, None], size=mri.shape[2:], mode='trilinear', align_corners=False).squeeze().cpu().numpy()
        heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
        return heatmap, pred

@torch.no_grad()
def get_attention_map(model, sample):
    mri = sample['mri'].unsqueeze(0).to(device)
    pet = sample['pet'].unsqueeze(0).to(device)
    clin = sample['clinical'].unsqueeze(0).to(device)
    pet_conf = sample['pet_confidence'].unsqueeze(0).to(device)

    _, mri_sp = model.mri_backbone(mri)
    _, pet_sp = model.pet_backbone(pet)
    _, attn = model.fusion(mri_sp, pet_sp, pet_conf)

    spatial_shape = mri_sp.shape[2:]
    attn_np = attn[0].cpu().numpy()
    attn_received = attn_np.mean(axis=0).reshape(spatial_shape)
    zoom = [s / a for s, a in zip(cfg.CROP_SHAPE, spatial_shape)]
    attn_up = scipy_zoom(attn_received, zoom, order=1)
    attn_up = (attn_up - attn_up.min()) / (attn_up.max() - attn_up.min() + 1e-8)
    return attn_up

gradcam_mri = GradCAM3D(best_fold_model, best_fold_model.mri_backbone.layer4)
demo_idx = 0
demo_sample = best_val_dataset[demo_idx]

mri = demo_sample['mri'].unsqueeze(0).to(device)
pet = demo_sample['pet'].unsqueeze(0).to(device)
clin = demo_sample['clinical'].unsqueeze(0).to(device)
pet_conf = demo_sample['pet_confidence'].unsqueeze(0).to(device)

heatmap, pred_cls = gradcam_mri.generate(mri, pet, clin, pet_conf)
attn_map = get_attention_map(best_fold_model, demo_sample)

mri_np = demo_sample['mri'][0].numpy()
D, H, W = mri_np.shape

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
mid = (D // 2, H // 2, W // 2)
axes[0].imshow(mri_np[:, :, mid[2]], cmap='gray', vmin=-1, vmax=1)
axes[0].set_title('MRI axial')
axes[0].axis('off')
axes[1].imshow(mri_np[:, :, mid[2]], cmap='gray', vmin=-1, vmax=1)
axes[1].imshow(heatmap[:, :, mid[2]], cmap='jet', alpha=0.45, vmin=0, vmax=1)
axes[1].set_title('Grad-CAM overlay')
axes[1].axis('off')
axes[2].imshow(attn_map[:, :, mid[2]], cmap='magma', vmin=0, vmax=1)
axes[2].set_title('Cross-attention map')
axes[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, 'xai_demo.png'), dpi=150)
plt.show()
print(f"Demo sample true={cfg.CLASS_NAMES[demo_sample['label'].item()]} | pred={cfg.CLASS_NAMES[pred_cls]}")

In [ ]:
# ============================================================
# SAVE FINAL REPORT
# ============================================================
final_report = {
    'kfold': fold_summaries,
    'mean_acc': float(np.mean([f['acc'] for f in fold_summaries])),
    'std_acc': float(np.std([f['acc'] for f in fold_summaries])),
    'mean_bal_acc': float(np.mean([f['bal_acc'] for f in fold_summaries])),
    'std_bal_acc': float(np.std([f['bal_acc'] for f in fold_summaries])),
    'best_fold': int(best_fold_id),
    'best_fold_bal_acc': float(best_fold_bal_acc),
    'mc_acc_best_fold': float(mc_acc),
    'mc_bal_acc_best_fold': float(mc_bal),
    'ece_before_temp': float(ece_before),
    'config': {
        'epochs': cfg.EPOCHS,
        'early_stopping_patience': cfg.EARLY_STOPPING_PATIENCE,
        'label_smoothing': cfg.LABEL_SMOOTHING,
        'reg_weight_start': cfg.REG_WEIGHT_START,
        'reg_weight_end': cfg.REG_WEIGHT_END,
        'confidence_penalty': cfg.CONFIDENCE_PENALTY,
        'n_folds': cfg.N_FOLDS,
        'freeze_backbone_epochs': cfg.FREEZE_BACKBONE_EPOCHS
    }
}

report_path = os.path.join(cfg.SAVE_DIR, 'updated_ad_classifier_report.json')
with open(report_path, 'w') as f:
    json.dump(final_report, f, indent=2)

print('Final report saved to:', report_path)
print('Notebook run complete.')